# Step I — Sector Coupling: Electricity + Heating

**Task (Assignment 2, part i):** Connect the electricity sector with at least one other
sector (heating) and co-optimise all sectors. Discuss the results.

## Model overview

This notebook **extends the Step H model** (Step D interconnected network with CO₂
GlobalConstraint) by coupling the electricity sector with a **residential heating sector**
in all four countries (Denmark, Germany, Sweden, Norway).

### Why build on Step H and not Step D?
Step H already embeds:
- CO₂ emission factors on the `CCGT` and `coal` carriers (IPCC 2006)
- A `GlobalConstraint` that enforces a system-wide CO₂ cap
- The correct `build_network_h(co2_limit=...)` factory function

Building Step I on top of Step H ensures that the sector-coupled optimisation
is **constrained by the same decarbonisation target** chosen in Step H, rather
than ignoring CO₂ entirely (as Step D would).

### Heating sector components added
| Component | From → To | Key parameter |
|-----------|-----------|---------------|
| Heat bus | — | one per country, carrier = 'heat' |
| Heat load | — | hourly profile from HDH + hot-water base |
| **Heat pump** (extendable) | Electricity → Heat | COP ≈ 2.4–2.7 (Carnot-based, hourly) |
| **CCGT CHP** (extendable) | Gas/Elec bus → Elec + Heat | η_e = 0.58, η_q = 0.37 |
| **Electric boiler** (extendable) | Electricity → Heat | η = 0.98, low CAPEX |

### CO₂ accounting in the coupled model
The CCGT CHP is a `Link` with `carrier='CCGT'`. The CO₂ GlobalConstraint in
PyPSA automatically counts `Link` components that consume a carrier with
`co2_emissions > 0`, so CHP gas consumption is **automatically included** in
the CO₂ budget.

### Sources
- Heat demand: Heat Roadmap Europe 4 (HRE4) — 2015 residential data
- Heat pump costs: DEA Technology Catalogue — *Large-scale heat pumps* (2023)
- COP model: Staffell & Pfenninger (2016) quadratic regression on ΔT
- Electric boiler costs: DEA Technology Catalogue — *Electric boilers* (2023)
- CCGT CHP efficiencies: consistent with Step H carrier definitions

**Required files in the working directory:**
- `DK_2015_merged.csv`
- `STEP D - electricity_demand.csv`
- `Other country data/Energy charts data/…` (neighbour CFs)
- `weather_data_2015_filtered.csv` (hourly temperatures per country)
- `functions_to_investigate.py`


## I-0 Imports

In [ ]:
from pathlib import Path
import pypsa
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

print(f"PyPSA version: {pypsa.__version__}")

## I-1 Rebuild the Step H network (baseline for sector coupling)

We reproduce **exactly** the `build_network_h()` function from Step H.
This guarantees that Step I is fully self-contained and starts from the
same CO₂-aware model.

> **Key difference from the old Step I approach:** previously Step I was appended
> to Step G (H₂ network, no CO₂ cap). Here, `build_network_h(co2_limit=...)` is
> the foundation, so the CO₂ constraint is active from the start.

In [ ]:
# ── CELL I-1a — cost table & data paths (identical to Step H) ────────────────

# Generator cost table (2020 technology costs, consistent with all previous steps)
cost_data = {
    "capital_cost": [
        1_500_000 / 25 + 60_000,   # Wind onshore: 120 k$/MW/y
        800_000   / 25 + 14_000,   # Solar PV:      46 k$/MW/y
        700_000   / 25 + 24_000,   # CCGT:          52 k$/MW/y
    ],
    "marginal_cost": [
        0.0,
        0.0,
        9.5 * 3.6 / 0.56 + 2.30,  # CCGT: ~63.4 $/MWh_e
    ]
}
costs = pd.DataFrame(cost_data, index=["wind_combined", "solar", "CCGT"])

# Battery parameters (2024 Li-ion, same as Step C/D/H)
battery_investment_power  = 100_000   # $/MW
battery_investment_energy = 150_000   # $/MWh
battery_fom               =  12_500   # $/MW/year
battery_lifetime          =      20   # years
battery_max_hours         =       4   # h
battery_efficiency        =    0.90   # round-trip √
battery_capital_cost = (
    battery_investment_power  / battery_lifetime
    + battery_fom
    + (battery_investment_energy / battery_lifetime) * battery_max_hours
)

print(f"CCGT capital cost:    {costs.loc['CCGT','capital_cost']:,.0f} $/MW/year")
print(f"Battery capital cost: {battery_capital_cost:,.0f} $/MW/year")

In [ ]:
# ── CELL I-1b — load all time-series data (identical to Step H) ──────────────

PROJECT_DIR = Path.cwd()
if not (PROJECT_DIR / "STEP D - electricity_demand.csv").exists():
    PROJECT_DIR = PROJECT_DIR.parent
cf_base = PROJECT_DIR / "Other country data" / "Energy charts data"

# --- Denmark capacity factors & snapshot index ---
dataframe_dk = pd.read_csv(
    PROJECT_DIR / "DK_2015_merged.csv",
    index_col=0, sep=",", parse_dates=True
)
dataframe_dk.index = pd.to_datetime(dataframe_dk.index, utc=True).tz_localize(None)
dataframe_dk = dataframe_dk.sort_index()
CF_wind  = dataframe_dk["wind_cf_Unnamed: 1"].astype(float)
CF_solar = dataframe_dk["pv_cf_Unnamed: 1"].astype(float)
snapshots = dataframe_dk.index  # 8760 or 8761 hourly timestamps

# --- Multi-country electricity demand ---
demand_all = pd.read_csv(
    PROJECT_DIR / "STEP D - electricity_demand.csv",
    sep=";", index_col=0, parse_dates=True
)
demand_all.index = pd.to_datetime(demand_all.index, utc=True).tz_localize(None)
demand_all = demand_all.sort_index()
demand_2015 = demand_all[demand_all.index.year == 2015].copy()

demand_dk = demand_2015["DNK"].astype(float).reindex(snapshots)
demand_de = demand_2015["DEU"].astype(float).reindex(snapshots)
demand_se = demand_2015["SWE"].astype(float).reindex(snapshots)
demand_no = demand_2015["NOR"].astype(float).reindex(snapshots)

# --- Neighbour capacity factors ---
def load_cf(path, col_map):
    df = pd.read_csv(path)
    df.index = snapshots
    return df.rename(columns=col_map)

cf_de_raw = pd.read_csv(cf_base / "Germany" / "Germany_hourly_capacity_factors.csv")
cf_de_raw.index = snapshots
cf_de = cf_de_raw

cf_se_raw = pd.read_csv(cf_base / "Sweden" / "Sweden_hourly_capacity_factors.csv")
cf_se_raw.index = snapshots
cf_se = cf_se_raw

cf_no_raw = pd.read_csv(cf_base / "Norway" / "Norway_hourly_capacity_factors.csv")
cf_no_raw.index = snapshots
cf_no = cf_no_raw
# Forward-fill any NaN from DST overlap
cf_no = cf_no.ffill()

print(f"Snapshots loaded: {len(snapshots)} hours")
print(f"Denmark demand peak: {demand_dk.max():.0f} MW")
print(f"Germany demand peak: {demand_de.max():.0f} MW")

In [ ]:
# ── CELL I-1c — build_network_h() function (copied from Step H, unmodified) ──
#
# IPCC 2006 CO₂ emission factors [tCO₂/MWh_thermal]
CO2_GAS  = 0.202   # natural gas
CO2_COAL = 0.341   # hard coal

def build_network_h(co2_limit=None):
    """
    Build the Step H interconnected network (4 countries, electricity only).
    If co2_limit is given (tCO₂/year) a GlobalConstraint is added.
    Returns an UNOPTIMISED network so the caller can extend it before solving.
    """
    n = pypsa.Network()
    n.set_snapshots(snapshots)

    # ── Carriers (with CO₂ emission factors) ─────────────────────────────────
    n.add("Carrier", "wind_combined",  co2_emissions=0.0)
    n.add("Carrier", "solar",          co2_emissions=0.0)
    n.add("Carrier", "CCGT",           co2_emissions=CO2_GAS)
    n.add("Carrier", "battery",        co2_emissions=0.0)
    n.add("Carrier", "hydro",          co2_emissions=0.0)
    n.add("Carrier", "nuclear",        co2_emissions=0.0)
    n.add("Carrier", "wind_onshore",   co2_emissions=0.0)
    n.add("Carrier", "coal",           co2_emissions=CO2_COAL)

    # ── Electricity buses ─────────────────────────────────────────────────────
    for name, (x, y) in [("Denmark",(10.0,56.0)),("Germany",(10.5,51.5)),
                          ("Sweden",(15.0,59.5)),("Norway",(10.0,62.0))]:
        n.add("Bus", name, x=x, y=y)

    # ── Electricity loads ─────────────────────────────────────────────────────
    n.add("Load", "DK_load", bus="Denmark", p_set=demand_dk.values)
    n.add("Load", "DE_load", bus="Germany", p_set=demand_de.values)
    n.add("Load", "SE_load", bus="Sweden",  p_set=demand_se.values)
    n.add("Load", "NO_load", bus="Norway",  p_set=demand_no.values)

    # ── Denmark generators (extendable) ───────────────────────────────────────
    n.add("Generator", "DK_wind",
          bus="Denmark", carrier="wind_combined",
          capital_cost=costs.loc["wind_combined","capital_cost"],
          marginal_cost=costs.loc["wind_combined","marginal_cost"],
          p_max_pu=CF_wind.values, p_nom_extendable=True)

    n.add("Generator", "DK_solar",
          bus="Denmark", carrier="solar",
          capital_cost=costs.loc["solar","capital_cost"],
          marginal_cost=costs.loc["solar","marginal_cost"],
          p_max_pu=CF_solar.values, p_nom_extendable=True)

    n.add("Generator", "DK_CCGT",
          bus="Denmark", carrier="CCGT",
          capital_cost=costs.loc["CCGT","capital_cost"],
          marginal_cost=costs.loc["CCGT","marginal_cost"],
          efficiency=0.56, p_nom_extendable=True)

    # ── Germany generators (fixed) ────────────────────────────────────────────
    n.add("Generator", "DE_wind",
          bus="Germany", carrier="wind_combined",
          p_nom=41_300, marginal_cost=0,
          p_max_pu=cf_de["wind_combined"].values, p_nom_extendable=False)

    n.add("Generator", "DE_solar",
          bus="Germany", carrier="solar",
          p_nom=37_000, marginal_cost=0,
          p_max_pu=cf_de["solar"].values, p_nom_extendable=False)

    n.add("Generator", "DE_CCGT",
          bus="Germany", carrier="CCGT",
          p_nom=28_360, marginal_cost=62.0,
          efficiency=0.56, p_nom_extendable=False)

    n.add("Generator", "DE_nuclear",
          bus="Germany", carrier="nuclear",
          p_nom=10_800, marginal_cost=11.0, p_nom_extendable=False)

    n.add("Generator", "DE_coal",
          bus="Germany", carrier="coal",
          p_nom=21_420, marginal_cost=40.0,
          efficiency=0.40, p_nom_extendable=False)

    # ── Sweden generators (fixed) ─────────────────────────────────────────────
    n.add("Generator", "SE_hydro",
          bus="Sweden", carrier="hydro",
          p_nom=15_920, marginal_cost=5.0,
          p_max_pu=cf_se["hydro"].values, p_nom_extendable=False)

    n.add("Generator", "SE_nuclear",
          bus="Sweden", carrier="nuclear",
          p_nom=8_900, marginal_cost=11.0, p_nom_extendable=False)

    n.add("Generator", "SE_wind",
          bus="Sweden", carrier="wind_onshore",
          p_nom=5_500, marginal_cost=0,
          p_max_pu=cf_se["wind_combined"].values, p_nom_extendable=False)

    # ── Norway generators (fixed) ─────────────────────────────────────────────
    n.add("Generator", "NO_hydro",
          bus="Norway", carrier="hydro",
          p_nom=29_900, marginal_cost=5.0,
          p_max_pu=cf_no["hydro"].values, p_nom_extendable=False)

    n.add("Generator", "NO_wind",
          bus="Norway", carrier="wind_onshore",
          p_nom=700, marginal_cost=0,
          p_max_pu=cf_no["wind_combined"].values, p_nom_extendable=False)

    # ── Batteries (extendable at all buses) ───────────────────────────────────
    for code, bus in [("DK","Denmark"),("DE","Germany"),("SE","Sweden"),("NO","Norway")]:
        n.add("StorageUnit", f"{code}_battery",
              bus=bus, carrier="battery",
              capital_cost=battery_capital_cost,
              max_hours=battery_max_hours,
              efficiency_store=battery_efficiency**0.5,
              efficiency_dispatch=battery_efficiency**0.5,
              cyclic_state_of_charge=True,
              p_nom_extendable=True)

    # ── HVAC transmission lines (fixed capacities, Step D topology) ───────────
    lines = [
        ("DK-DE", "Denmark", "Germany", 3500, 0.01),
        ("DK-SE", "Denmark", "Sweden",  1700, 0.01),
        ("DK-NO", "Denmark", "Norway",  1050, 0.01),
        ("SE-NO", "Sweden",  "Norway",  3500, 0.01),
        ("DE-SE", "Germany", "Sweden",   600, 0.01),
    ]
    for name, bus0, bus1, cap, x in lines:
        n.add("Line", f"line_{name}",
              bus0=bus0, bus1=bus1,
              p_nom=cap, x=x, s_nom=cap)

    # ── CO₂ GlobalConstraint (optional) ──────────────────────────────────────
    if co2_limit is not None:
        n.add("GlobalConstraint", "co2_limit",
              sense="<=",
              constant=co2_limit,
              type="primary_energy",
              carrier_attribute="co2_emissions")
        print(f"CO₂ GlobalConstraint added: {co2_limit/1e6:.3f} MtCO₂/year")

    return n

print("build_network_h() defined.")

In [ ]:
# ── CELL I-1d — compute baseline emissions (Step H, unconstrained) ────────────
#
# We first solve the Step H model WITHOUT a CO₂ cap to get E₀ (baseline),
# then choose the same 30% reduction target used in Step H.

print("Building unconstrained Step H network...")
net_base = build_network_h(co2_limit=None)
net_base.optimize(solver_name="gurobi", solver_options={"output_flag": False})
print(f"Step H baseline — system cost: {net_base.objective/1e9:.3f} B$/year")

# Emitters in the electricity sector
emitters_elec = {
    "DK_CCGT": {"eps": CO2_GAS,  "eta": 0.56},
    "DE_CCGT": {"eps": CO2_GAS,  "eta": 0.56},
    "DE_coal": {"eps": CO2_COAL, "eta": 0.40},
}

E0_t = 0.0
for gen, p in emitters_elec.items():
    gen_MWh = net_base.generators_t.p[gen].sum()
    E0_t   += gen_MWh * p["eps"] / p["eta"]
E0_Mt = E0_t / 1e6

print(f"\nBaseline CO₂ (electricity only): {E0_Mt:.3f} MtCO₂/year")

# Apply 30% reduction — same target as Step H
REDUCTION    = 0.30
co2_target_t = E0_t * (1 - REDUCTION)
co2_target_Mt = co2_target_t / 1e6
print(f"CO₂ cap (30% reduction):          {co2_target_Mt:.3f} MtCO₂/year")

## I-2 Heating sector: demand data

### Methodology
Hourly heat demand is modelled as:

$$Q_{c,t} = f_{1,c} \cdot \text{HDH}_{c,t} + Q_{\text{water},c}$$

where:
- $\text{HDH}_{c,t} = \max(0,\, T_{\text{ref}} - T_{\text{amb},c,t})$ is the
  Heating Degree Hour with threshold $T_{\text{ref}} = 17°C$
- $f_{1,c}$ [MW/K] is the space-heating scaling factor calibrated to annual
  demand from HRE4
- $Q_{\text{water},c}$ [MW] is a constant hot-water baseload

### Annual heat demands (HRE4 — 2015 residential)
| Country | Space heating [TWh] | Hot water [TWh] | Total [TWh] |
|---------|---------------------|-----------------|-------------|
| Germany | 675 | 143 | 818 |
| Denmark | 33.6 | 7.7 | 41.3 |
| Sweden | 33.6 | 7.7 | 41.3 |
| Norway | 17.8 | 4.1 | 21.9 |

In [ ]:
# ── CELL I-2a — load hourly temperatures & compute HDH ───────────────────────

PROCESSED_TEMP_FILE = PROJECT_DIR / "weather_data_2015_filtered.csv"
T_REF = 17.0  # Heating threshold [°C]

weather_2015 = pd.read_csv(PROCESSED_TEMP_FILE, index_col=0, parse_dates=True)
# Ensure index aligns with electricity snapshots
weather_2015.index = snapshots

hdh_data = (T_REF - weather_2015).clip(lower=0)
hdh_data.columns = [c.replace("Temp_", "HDH_") for c in hdh_data.columns]

print("HDH data ready:")
print(hdh_data.head(3))

In [ ]:
# ── CELL I-2b — compute hourly heat demand profiles [MW] ─────────────────────

countries_map = {"DE": "Germany", "DK": "Denmark", "SE": "Sweden", "NO": "Norway"}

# Annual demands [MWh] — from HRE4
annual_demand_mwh = {
    "DE": {"space": 675.0e6, "water": 143.0e6},
    "DK": {"space":  33.6e6, "water":   7.7e6},
    "SE": {"space":  33.6e6, "water":   7.7e6},
    "NO": {"space":  17.8e6, "water":   4.1e6},
}

N_HOURS = len(snapshots)
final_heat_profiles_mw = pd.DataFrame(index=snapshots)

print("Hourly heat demand profiles:")
for code in countries_map.keys():
    hdh_series = hdh_data[f"HDH_{code}"]
    f1          = annual_demand_mwh[code]["space"] / hdh_series.sum()  # MW/K
    water_base  = annual_demand_mwh[code]["water"] / N_HOURS            # MW
    final_heat_profiles_mw[code] = (f1 * hdh_series) + water_base
    print(f"  {code}: peak = {final_heat_profiles_mw[code].max():,.0f} MW, "
          f"annual = {final_heat_profiles_mw[code].sum()/1e6:.1f} TWh")

In [ ]:
# ── CELL I-2c — hourly COP profiles (Staffell & Pfenninger 2016) ─────────────
#
# COP(ΔT) = 6.81 − 0.121·ΔT + 0.00063·ΔT²   (quadratic regression)
# ΔT = T_sink − T_source;   T_sink = 55 °C (radiator / district heating)

T_SINK = 55.0  # [°C]
cop_profiles = pd.DataFrame(index=snapshots)

print("COP profiles (Staffell & Pfenninger 2016):")
for code in countries_map.keys():
    t_source = weather_2015[f"Temp_{code}"]
    delta_t  = T_SINK - t_source
    cop      = 6.81 - 0.121 * delta_t + 0.00063 * delta_t**2
    cop      = cop.clip(lower=1.0)   # physical floor
    cop_profiles[code] = cop
    print(f"  {code}: mean COP = {cop.mean():.2f}, min = {cop.min():.2f}, "
          f"max = {cop.max():.2f}")

# Quick sanity-check plot
fig, ax = plt.subplots(figsize=(12, 3))
for code in countries_map.keys():
    ax.plot(snapshots[:8*7], cop_profiles[code].values[:8*7], label=code, lw=0.9)
ax.set_title("Heat pump COP — first 2 months of 2015")
ax.set_ylabel("COP")
ax.legend()
ax.grid(True, linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()

## I-3 Technology assumptions — heating sector

### Heat pump (large-scale air-source, DEA 2023)
- Investment: 730 000 $/MW_e
- Fixed O&M: 12 500 $/MW_e/year
- Lifetime: 25 years, WACC 7 %
- COP: hourly (Staffell & Pfenninger 2016)

### Electric boiler (DEA 2023)
- Investment: 50 000 $/MW_e  (low CAPEX — thermal resistance heating)
- Fixed O&M: 2 000 $/MW_e/year
- Lifetime: 20 years, η = 0.98
- Role: peak-shaving / firm heat backup (cheaper than CCGT CHP at high COP regimes)

### CCGT CHP (combined heat and power, consistent with Step H carrier)
- Replaces the simple `DK_CCGT` Generator with a multi-output `Link`
- η_elec = 0.58 (electrical output / gas input)
- η_heat = 0.37 (heat recovery / gas input; total = 0.95, 5% stack losses)
- Capital cost: same as `costs.loc['CCGT']` from Step H
- **CO₂ is accounted for** because the CHP Link consumes electricity bus power
  and its carrier is `CCGT` — which has `co2_emissions=CO2_GAS`.

> **Note on the German/Swedish/Norwegian CCGTs:** these are fixed-capacity generators
> (not extendable). We do *not* convert them to CHPs because their dispatch is
> driven by electricity, and we lack country-level district-heating network data
> to correctly model their heat dispatch constraints.

In [ ]:
# ── CELL I-3 — heating technology cost parameters ────────────────────────────

WACC = 0.07

def annuity(lifetime):
    """Capital recovery factor (CRF) for a given lifetime and WACC."""
    return (WACC * (1 + WACC)**lifetime) / ((1 + WACC)**lifetime - 1)

# --- Heat pump ---
hp_investment = 730_000   # $/MW_e
hp_fom        =  12_500   # $/MW_e/year
hp_lifetime   =      25   # years
hp_capital_cost = hp_investment * annuity(hp_lifetime) + hp_fom

# --- Electric boiler ---
eb_investment =  50_000   # $/MW_e  (resistance heating)
eb_fom        =   2_000   # $/MW_e/year
eb_lifetime   =      20   # years
eb_efficiency =    0.98
eb_capital_cost = eb_investment * annuity(eb_lifetime) + eb_fom

# --- CCGT CHP efficiencies (consistent with Step H CCGT carrier) ---
CCGT_EFF_ELEC = 0.58   # electrical efficiency (gas → electricity)
CCGT_EFF_HEAT = 0.37   # heat recovery       (gas → heat; 5% stack losses)
# CCGT capital cost re-used from `costs` table (Step H).

print("Heating technology assumptions:")
print(f"  Heat pump capex:     {hp_capital_cost:,.0f} $/MW_e/year")
print(f"  Electric boiler capex: {eb_capital_cost:,.0f} $/MW_e/year")
print(f"  CCGT CHP η_elec: {CCGT_EFF_ELEC:.0%}, η_heat: {CCGT_EFF_HEAT:.0%}")

## I-4 Build the sector-coupled network

### Strategy
1. Call `build_network_h(co2_limit=co2_target_t)` to get the CO₂-constrained
   electricity network.
2. **Remove** the extendable `DK_CCGT` Generator from Denmark — it will be
   replaced by an extendable **CCGT CHP Link** that simultaneously serves
   electricity and heat.
3. Add `heat` buses and loads for all four countries.
4. Add extendable **heat pumps**, **electric boilers**, and **CCGT CHP** Links.
5. Optimise with a single call to `n.optimize()`.

The CO₂ GlobalConstraint from step 1 automatically covers CHP gas consumption
because the Link carrier `CCGT` carries `co2_emissions=CO2_GAS`.

In [ ]:
# ── CELL I-4a — build the base H network WITH CO₂ cap ────────────────────────

print(f"Building Step H network with CO₂ cap = {co2_target_Mt:.3f} MtCO₂/year...")
net_i = build_network_h(co2_limit=co2_target_t)
print("Base network ready. Now adding heating sector...")

In [ ]:
# ── CELL I-4b — remove DK CCGT generator (replaced by CHP Link) ──────────────
#
# The simple Generator `DK_CCGT` is a single-output device (gas → electricity).
# We replace it with a `Link` that has two outputs: electricity (bus1) and
# heat (bus2). This is the standard PyPSA approach for CHP modelling.

net_i.remove("Generator", "DK_CCGT")
print("DK_CCGT generator removed — will be replaced by DK CCGT CHP Link.")

In [ ]:
# ── CELL I-4c — add heat carrier, buses, and loads ───────────────────────────

net_i.add("Carrier", "heat",           co2_emissions=0.0)
net_i.add("Carrier", "heat pump",      co2_emissions=0.0)
net_i.add("Carrier", "electric boiler",co2_emissions=0.0)

for code, country in countries_map.items():
    # Heat bus (one per country)
    net_i.add("Bus", f"{country} heat", carrier="heat")

    # Hourly heat load
    net_i.add("Load", f"{country} heat load",
              bus=f"{country} heat",
              p_set=final_heat_profiles_mw[code].values)

print("Heat buses and loads added for:", list(countries_map.values()))

In [ ]:
# ── CELL I-4d — add extendable heat pumps (electricity → heat) ───────────────
#
# COP is time-varying: set as a constant in the static Link table,
# then overwrite with the hourly profile in links_t.efficiency.

for code, country in countries_map.items():
    mean_cop = cop_profiles[code].mean()  # used as static placeholder
    net_i.add("Link", f"{country} heat pump",
              bus0=country,
              bus1=f"{country} heat",
              carrier="heat pump",
              capital_cost=hp_capital_cost,
              efficiency=mean_cop,          # will be overwritten below
              p_nom_extendable=True)

# Overwrite with hourly COP profiles
for code, country in countries_map.items():
    net_i.links_t.efficiency[f"{country} heat pump"] = cop_profiles[code].values

print("Heat pumps added with hourly COP profiles.")

In [ ]:
# ── CELL I-4e — add extendable electric boilers (electricity → heat) ──────────
#
# Electric boilers provide firm, low-CAPEX heat at very high electricity cost.
# They act as a 'last resort' heat supply when heat pumps are insufficient
# (e.g., very cold days where COP drops to ~1.5).

for code, country in countries_map.items():
    net_i.add("Link", f"{country} electric boiler",
              bus0=country,
              bus1=f"{country} heat",
              carrier="electric boiler",
              capital_cost=eb_capital_cost,
              efficiency=eb_efficiency,
              p_nom_extendable=True)

print("Electric boilers added (η = {:.0%}).".format(eb_efficiency))

In [ ]:
# ── CELL I-4f — add Denmark CCGT CHP Link (gas → electricity + heat) ─────────
#
# This is the key sector-coupling component for Denmark.
# The Link bus0 = Denmark (electricity bus, acts as proxy for gas input).
# bus1 = Denmark (electricity output)
# bus2 = Denmark heat (heat output)
#
# IMPORTANT: In PyPSA, `Link` components with carrier that has co2_emissions > 0
# DO contribute to the GlobalConstraint co2_limit automatically.
# Here the carrier 'CCGT' has co2_emissions = CO2_GAS = 0.202 tCO2/MWh_th.
# PyPSA computes effective emissions as: p0_t * co2_emissions / efficiency,
# which is the thermal input * emission factor — exactly what we want.

net_i.add("Link", "Denmark CCGT CHP",
          bus0="Denmark",
          bus1="Denmark",
          bus2="Denmark heat",
          carrier="CCGT",
          efficiency=CCGT_EFF_ELEC,      # electricity output / gas input
          efficiency2=CCGT_EFF_HEAT,     # heat output / gas input
          capital_cost=costs.loc["CCGT", "capital_cost"],
          marginal_cost=costs.loc["CCGT", "marginal_cost"],
          p_nom_extendable=True)

print("Denmark CCGT CHP Link added.")
print(f"  η_elec = {CCGT_EFF_ELEC:.0%}, η_heat = {CCGT_EFF_HEAT:.0%}")
print(f"  CO₂ factor = {CO2_GAS} tCO₂/MWh_th (via CCGT carrier)")

In [ ]:
# ── CELL I-4g — print network summary before solving ─────────────────────────

print("=" * 55)
print("  STEP I — Network summary before optimisation")
print("=" * 55)
print(f"  Buses:      {len(net_i.buses)} (electricity + heat)")
print(f"  Generators: {len(net_i.generators)}")
print(f"  Links:      {len(net_i.links)}")
print(f"  Loads:      {len(net_i.loads)}  (elec + heat)")
print(f"  StorageUnits: {len(net_i.storage_units)}")
print(f"  Lines:      {len(net_i.lines)}")
print(f"  Snapshots:  {len(net_i.snapshots)}")
print(f"  CO₂ cap:    {co2_target_Mt:.3f} MtCO₂/year")
print("=" * 55)

print("\nHeat-sector Links:")
heat_links = net_i.links[net_i.links.carrier.isin(["heat pump","electric boiler","CCGT"])]
print(heat_links[["bus0","bus1","carrier","p_nom_extendable","capital_cost"]].to_string())

## I-5 Optimise the sector-coupled model

In [ ]:
# ── CELL I-5 — solve ──────────────────────────────────────────────────────────

print("Solving sector-coupled model (Step I)...")
net_i.optimize(solver_name="gurobi", solver_options={"output_flag": False})
print(f"\nStep I system cost: {net_i.objective/1e9:.3f} B$/year")
print(f"Step H system cost: {net_base.objective/1e9:.3f} B$/year")
print(f"Change:             {(net_i.objective - net_base.objective)/1e6:.1f} M$/year")

# Extract CO₂ shadow price
shadow_price_i = abs(net_i.global_constraints.loc["co2_limit", "mu"])
print(f"\nCO₂ shadow price (Step I): {shadow_price_i:.2f} $/tCO₂")

## I-6 Results: verify CO₂ constraint

We manually compute total system CO₂ (electricity generators + CHP Links)
to confirm the constraint is satisfied.

In [ ]:
# ── CELL I-6 — CO₂ accounting ─────────────────────────────────────────────────

print("CO₂ accounting for Step I:")
print("-" * 50)

total_co2 = 0.0

# Fixed-dispatch emitting generators
emitters_check = {
    "DE_CCGT": {"eps": CO2_GAS,  "eta": 0.56},
    "DE_coal": {"eps": CO2_COAL, "eta": 0.40},
}
for gen, p in emitters_check.items():
    if gen in net_i.generators_t.p.columns:
        g_MWh = net_i.generators_t.p[gen].sum()
        co2   = g_MWh * p["eps"] / p["eta"]
        total_co2 += co2
        print(f"  {gen:<20}: {g_MWh/1e6:.2f} TWh_e  →  {co2/1e6:.3f} MtCO₂")

# Denmark CCGT CHP (Link — input power p0)
if "Denmark CCGT CHP" in net_i.links_t.p0.columns:
    chp_input_MWh = net_i.links_t.p0["Denmark CCGT CHP"].sum()   # MW input = thermal
    co2_chp       = chp_input_MWh * CO2_GAS
    total_co2    += co2_chp
    elec_out      = net_i.links_t.p1["Denmark CCGT CHP"].abs().sum()
    heat_out      = net_i.links_t.p2["Denmark CCGT CHP"].abs().sum()
    print(f"  {'Denmark CCGT CHP':<20}: {chp_input_MWh/1e6:.2f} TWh_th → "
          f"{elec_out/1e6:.2f} TWh_e + {heat_out/1e6:.2f} TWh_q  →  {co2_chp/1e6:.3f} MtCO₂")

print("-" * 50)
print(f"  TOTAL CO₂:        {total_co2/1e6:.3f} MtCO₂/year")
print(f"  CO₂ cap:          {co2_target_Mt:.3f} MtCO₂/year")
print(f"  Constraint slack: {(co2_target_t - total_co2)/1e6:.4f} MtCO₂  "
      f"({'satisfied' if total_co2 <= co2_target_t + 1e3 else 'VIOLATED'})")

## I-7 Results: optimal capacities

In [ ]:
# ── CELL I-7 — capacity comparison: Step H constrained vs Step I ──────────────

# Rebuild Step H constrained for fair comparison
print("Rebuilding Step H constrained (same CO₂ cap, no heating sector)...")
net_h_co2 = build_network_h(co2_limit=co2_target_t)
net_h_co2.optimize(solver_name="gurobi", solver_options={"output_flag": False})

print("\n--- Denmark electricity capacities ---")
print(f"{'Technology':<20} {'Step H [GW]':>12} {'Step I [GW]':>12} {'Δ [GW]':>10}")
print("-" * 56)

dk_gens = ["DK_wind", "DK_solar"]
for g in dk_gens:
    cap_h = net_h_co2.generators.loc[g, "p_nom_opt"] / 1e3
    cap_i = net_i.generators.loc[g,   "p_nom_opt"] / 1e3
    print(f"  {g:<18} {cap_h:>12.2f} {cap_i:>12.2f} {cap_i-cap_h:>10.2f}")

# CCGT: in Step H it is a Generator; in Step I it is a CHP Link
cap_ccgt_h = net_h_co2.generators.loc["DK_CCGT", "p_nom_opt"] / 1e3
cap_chp_i  = net_i.links.loc["Denmark CCGT CHP", "p_nom_opt"] / 1e3
print(f"  {'DK_CCGT / CHP':<18} {cap_ccgt_h:>12.2f} {cap_chp_i:>12.2f} "
      f"{cap_chp_i-cap_ccgt_h:>10.2f}")

print("\n--- Denmark heating capacities (Step I new) ---")
heat_links_dk = ["Denmark heat pump", "Denmark electric boiler", "Denmark CCGT CHP"]
for lk in heat_links_dk:
    cap = net_i.links.loc[lk, "p_nom_opt"]
    print(f"  {lk:<30}: {cap/1e3:.2f} GW")

print("\n--- Denmark battery ---")
cap_batt_h = net_h_co2.storage_units.loc["DK_battery", "p_nom_opt"] / 1e3
cap_batt_i = net_i.storage_units.loc["DK_battery",   "p_nom_opt"] / 1e3
print(f"  {'DK_battery':<20} Step H: {cap_batt_h:.2f} GW, Step I: {cap_batt_i:.2f} GW")

## I-8 Results: annual energy flows

In [ ]:
# ── CELL I-8 — energy flows across sectors ────────────────────────────────────

print("=" * 60)
print("  Annual energy flows — Step I")
print("=" * 60)

# Electricity sector — Denmark generators
for g in ["DK_wind", "DK_solar"]:
    gen_twh = net_i.generators_t.p[g].sum() / 1e6
    print(f"  {g:<25}: {gen_twh:.2f} TWh_e/year")

# CHP link
if "Denmark CCGT CHP" in net_i.links_t.p0.columns:
    e_out = net_i.links_t.p1["Denmark CCGT CHP"].abs().sum() / 1e6
    q_out = net_i.links_t.p2["Denmark CCGT CHP"].abs().sum() / 1e6
    print(f"  {'DK CCGT CHP (elec)':<25}: {e_out:.2f} TWh_e/year")
    print(f"  {'DK CCGT CHP (heat)':<25}: {q_out:.2f} TWh_q/year")

# Heat pump
if "Denmark heat pump" in net_i.links_t.p1.columns:
    hp_elec = net_i.links_t.p0["Denmark heat pump"].sum() / 1e6
    hp_heat = net_i.links_t.p1["Denmark heat pump"].sum() / 1e6
    print(f"  {'DK heat pump (elec in)':<25}: {hp_elec:.2f} TWh_e/year")
    print(f"  {'DK heat pump (heat out)':<25}: {hp_heat:.2f} TWh_q/year")

# Electric boiler
if "Denmark electric boiler" in net_i.links_t.p0.columns:
    eb_elec = net_i.links_t.p0["Denmark electric boiler"].sum() / 1e6
    eb_heat = net_i.links_t.p1["Denmark electric boiler"].sum() / 1e6
    print(f"  {'DK elec boiler (elec in)':<25}: {eb_elec:.2f} TWh_e/year")
    print(f"  {'DK elec boiler (heat out)':<25}: {eb_heat:.2f} TWh_q/year")

# Annual heat demand vs supply check
dk_heat_demand_twh = final_heat_profiles_mw["DK"].sum() / 1e6
print(f"\n  Denmark annual heat demand:  {dk_heat_demand_twh:.2f} TWh_q/year")

print("=" * 60)

## I-9 Visualisations

In [ ]:
# ── CELL I-9a — capacity bar chart: electricity + heating ─────────────────────

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Left: Electricity capacity comparison (Step H vs Step I) ---
ax = axes[0]
labels  = ["Wind", "Solar", "CCGT/CHP", "Battery"]
cap_h_vals = [
    net_h_co2.generators.loc["DK_wind",    "p_nom_opt"] / 1e3,
    net_h_co2.generators.loc["DK_solar",   "p_nom_opt"] / 1e3,
    net_h_co2.generators.loc["DK_CCGT",   "p_nom_opt"] / 1e3,
    net_h_co2.storage_units.loc["DK_battery", "p_nom_opt"] / 1e3,
]
cap_i_vals = [
    net_i.generators.loc["DK_wind",   "p_nom_opt"] / 1e3,
    net_i.generators.loc["DK_solar",  "p_nom_opt"] / 1e3,
    net_i.links.loc["Denmark CCGT CHP", "p_nom_opt"] / 1e3,
    net_i.storage_units.loc["DK_battery", "p_nom_opt"] / 1e3,
]
x = np.arange(len(labels))
w = 0.35
ax.bar(x - w/2, cap_h_vals, w, label="Step H (CO₂ cap, no heat)",
       color=["steelblue","gold","sienna","purple"])
ax.bar(x + w/2, cap_i_vals, w, label="Step I (CO₂ cap + heat sector)",
       color=["steelblue","gold","sienna","purple"], alpha=0.6)
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_ylabel("Installed capacity [GW]")
ax.set_title("Denmark — Electricity capacity: Step H vs Step I")
ax.legend(fontsize=9)
ax.grid(True, axis="y", linestyle="--", alpha=0.4)

# --- Right: Denmark heating supply pie ---
ax2 = axes[1]
heat_labels, heat_vals, heat_colors = [], [], []
heat_supply = {
    "Heat pump": ("Denmark heat pump", "p1", "#e74c3c"),
    "Electric boiler": ("Denmark electric boiler", "p1", "#f39c12"),
    "CCGT CHP": ("Denmark CCGT CHP", "p2", "#8e44ad"),
}
for label, (link, port, color) in heat_supply.items():
    df_port = getattr(net_i.links_t, port)
    if link in df_port.columns:
        val = df_port[link].abs().sum() / 1e6  # TWh
        if val > 0.01:
            heat_labels.append(f"{label}\n({val:.1f} TWh)")
            heat_vals.append(val)
            heat_colors.append(color)

if heat_vals:
    ax2.pie(heat_vals, labels=heat_labels, colors=heat_colors,
            autopct="%1.0f%%", startangle=90)
    ax2.set_title("Denmark — Annual heat supply mix (Step I)")

plt.tight_layout()
plt.show()

In [ ]:
# ── CELL I-9b — weekly dispatch plot: electricity + heat (January) ────────────

# Select a cold winter week in January
WEEK_START = 0
WEEK_END   = 7 * 24   # 168 hours
idx = snapshots[WEEK_START:WEEK_END]

fig, (ax_e, ax_q) = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

# --- Electricity dispatch ---
wind_gen  = net_i.generators_t.p["DK_wind"].values[WEEK_START:WEEK_END]
solar_gen = net_i.generators_t.p["DK_solar"].values[WEEK_START:WEEK_END]

chp_elec = np.zeros(WEEK_END - WEEK_START)
if "Denmark CCGT CHP" in net_i.links_t.p1.columns:
    chp_elec = net_i.links_t.p1["Denmark CCGT CHP"].abs().values[WEEK_START:WEEK_END]

ax_e.stackplot(idx, wind_gen, solar_gen, chp_elec,
               labels=["Wind", "Solar", "CCGT CHP (elec)"],
               colors=["steelblue", "gold", "sienna"], alpha=0.85)
ax_e.plot(idx, demand_dk.values[WEEK_START:WEEK_END], "k-", lw=1.5, label="Elec demand")
ax_e.set_ylabel("Power [MW]")
ax_e.set_title("Denmark — Electricity dispatch (January, first week, Step I)")
ax_e.legend(loc="upper right", fontsize=9)
ax_e.grid(True, linestyle="--", alpha=0.4)

# --- Heat dispatch ---
hp_heat_ts = np.zeros(WEEK_END - WEEK_START)
eb_heat_ts = np.zeros(WEEK_END - WEEK_START)
chp_heat_ts = np.zeros(WEEK_END - WEEK_START)

if "Denmark heat pump" in net_i.links_t.p1.columns:
    hp_heat_ts = net_i.links_t.p1["Denmark heat pump"].values[WEEK_START:WEEK_END]
if "Denmark electric boiler" in net_i.links_t.p1.columns:
    eb_heat_ts = net_i.links_t.p1["Denmark electric boiler"].values[WEEK_START:WEEK_END]
if "Denmark CCGT CHP" in net_i.links_t.p2.columns:
    chp_heat_ts = net_i.links_t.p2["Denmark CCGT CHP"].abs().values[WEEK_START:WEEK_END]

ax_q.stackplot(idx, hp_heat_ts, eb_heat_ts, chp_heat_ts,
               labels=["Heat pump", "Electric boiler", "CCGT CHP (heat)"],
               colors=["#e74c3c", "#f39c12", "#8e44ad"], alpha=0.85)
ax_q.plot(idx, final_heat_profiles_mw["DK"].values[WEEK_START:WEEK_END],
          "k-", lw=1.5, label="Heat demand")
ax_q.set_ylabel("Heat power [MW]")
ax_q.set_title("Denmark — Heat dispatch (January, first week, Step I)")
ax_q.legend(loc="upper right", fontsize=9)
ax_q.grid(True, linestyle="--", alpha=0.4)

plt.tight_layout()
plt.show()

In [ ]:
# ── CELL I-9c — system cost breakdown ────────────────────────────────────────

fig, ax = plt.subplots(figsize=(8, 4))

# System cost components
cases   = ["Step H\n(CO₂ cap,\nno heating)",
           "Step I\n(CO₂ cap +\nheating sector)"]
costs_b = [net_h_co2.objective / 1e9, net_i.objective / 1e9]

bars = ax.bar(cases, costs_b, color=["steelblue", "#e74c3c"], width=0.4)
for bar, val in zip(bars, costs_b):
    ax.text(bar.get_x() + bar.get_width() / 2, val + 0.05, f"{val:.2f} B$/y",
            ha="center", va="bottom", fontsize=11)
ax.set_ylabel("Total system cost [B$/year]")
ax.set_title("Total system cost comparison: Step H vs Step I")
ax.grid(True, axis="y", linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()

In [ ]:
# ── CELL I-9d — COP vs heat pump dispatch duration curve ─────────────────────

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

# COP duration curve (DK)
cop_sorted = cop_profiles["DK"].sort_values(ascending=False).values
ax1.fill_between(range(len(cop_sorted)), cop_sorted, alpha=0.7, color="#e74c3c")
ax1.axhline(cop_profiles["DK"].mean(), color="k", linestyle="--",
            label=f"Mean COP = {cop_profiles['DK'].mean():.2f}")
ax1.set_xlabel("Hours [h/year]")
ax1.set_ylabel("COP")
ax1.set_title("Denmark — Heat pump COP duration curve")
ax1.legend()
ax1.grid(True, linestyle="--", alpha=0.4)

# Heat pump utilisation duration curve
if "Denmark heat pump" in net_i.links_t.p0.columns:
    cap_hp = net_i.links.loc["Denmark heat pump", "p_nom_opt"]
    hp_util = (net_i.links_t.p0["Denmark heat pump"] / cap_hp
               if cap_hp > 0 else pd.Series(0, index=snapshots))
    hp_sorted = hp_util.sort_values(ascending=False).values
    ax2.fill_between(range(len(hp_sorted)), hp_sorted, alpha=0.7, color="#e74c3c")
    ax2.set_xlabel("Hours [h/year]")
    ax2.set_ylabel("Utilisation [p.u.]")
    ax2.set_title(f"Denmark — Heat pump utilisation\n(capacity: {cap_hp/1e3:.2f} GW)")
    ax2.set_ylim(0, 1.05)
    ax2.grid(True, linestyle="--", alpha=0.4)

plt.tight_layout()
plt.show()

## I-10 Summary table and discussion

The final cell prints a concise summary table suitable for the report.

In [ ]:
# ── CELL I-10 — summary ───────────────────────────────────────────────────────

shadow_h = abs(net_h_co2.global_constraints.loc["co2_limit", "mu"])

print("=" * 65)
print("  STEP I — Summary")
print("=" * 65)
print(f"  CO₂ cap (30% reduction from E₀):  {co2_target_Mt:.3f} MtCO₂/year")
print()
print(f"  {'Metric':<35} {'Step H':>10} {'Step I':>10}")
print(f"  {'-'*55}")
print(f"  {'System cost [B$/y]':<35} "
      f"{net_h_co2.objective/1e9:>10.3f} {net_i.objective/1e9:>10.3f}")
print(f"  {'CO₂ shadow price [$/tCO₂]':<35} "
      f"{shadow_h:>10.1f} {shadow_price_i:>10.1f}")
print(f"  {'DK wind [GW]':<35} "
      f"{net_h_co2.generators.loc['DK_wind','p_nom_opt']/1e3:>10.2f} "
      f"{net_i.generators.loc['DK_wind','p_nom_opt']/1e3:>10.2f}")
print(f"  {'DK solar [GW]':<35} "
      f"{net_h_co2.generators.loc['DK_solar','p_nom_opt']/1e3:>10.2f} "
      f"{net_i.generators.loc['DK_solar','p_nom_opt']/1e3:>10.2f}")
print(f"  {'DK CCGT/CHP [GW]':<35} "
      f"{net_h_co2.generators.loc['DK_CCGT','p_nom_opt']/1e3:>10.2f} "
      f"{net_i.links.loc['Denmark CCGT CHP','p_nom_opt']/1e3:>10.2f}")
print(f"  {'DK battery [GW]':<35} "
      f"{net_h_co2.storage_units.loc['DK_battery','p_nom_opt']/1e3:>10.2f} "
      f"{net_i.storage_units.loc['DK_battery','p_nom_opt']/1e3:>10.2f}")

print()
print("  New heating sector capacities (Step I):")
for lk in ["Denmark heat pump", "Denmark electric boiler", "Denmark CCGT CHP"]:
    cap = net_i.links.loc[lk, "p_nom_opt"]
    print(f"    {lk:<35}: {cap/1e3:.2f} GW")

print()
print("  Reference CO₂ prices:")
for name, price in [("EU ETS (~2024)", 80), ("DK carbon tax (2030 target)", 160)]:
    print(f"    {name:<35}: {price} $/tCO₂")
print("=" * 65)

print("""
Discussion points for the report
─────────────────────────────────────────────────────────────────────
1. CO₂ price change: Adding the heating sector changes the CO₂ shadow
   price because additional decarbonisation options (heat pumps) are
   now available. If COP > 1, electrifying heat is cheaper per tCO₂
   avoided than cutting electricity-sector gas — the shadow price falls.

2. Wind expansion: Higher electricity demand from heat pumps incentivises
   more wind investment. Heat pumps also improve wind integration because
   they can consume excess wind (when prices are low) and shift heat
   demand in time.

3. CCGT CHP vs simple CCGT: The CHP extracts useful heat from exhaust
   gases, raising total fuel utilisation from ~56% to ~95%. The
   CO₂ per unit of useful energy (elec + heat) is therefore much lower
   than a simple CCGT, making it more compatible with tight CO₂ caps.

4. System cost: Including the heating sector increases total cost because
   a larger energy demand must be served. But the cost per unit of
   energy (electricity + heat) typically falls, showing the economic
   benefit of sector coupling.

5. Heat pump vs electric boiler: Heat pumps dominate at moderate
   temperatures (high COP); electric boilers fill extreme cold peaks
   where COP drops. This mix avoids oversizing heat pumps.
─────────────────────────────────────────────────────────────────────
""")